In [3]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [4]:
df = pd.read_csv('crop_yield.csv')

In [5]:
# cat_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]
# le = LabelEncoder()
# for col in cat_cols:
#     df[col] = le.fit_transform(df[col].astype(str))

# df.head()

cat_cols = ["Region", "Soil_Type", "Crop", "Weather_Condition"]
num_cols  = ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest']
bool_cols = ['Fertilizer_Used', 'Irrigation_Used']

# df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Verify
df.head()
# print(df.shape)
# print(df.dtypes)

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251


In [67]:
# Convert booleans directly to int
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)

# Verify — call .unique() on each column separately
print(df["Fertilizer_Used"].unique())
print(df["Irrigation_Used"].unique())
print(df[["Fertilizer_Used", "Irrigation_Used"]].dtypes)

df.head()

[0 1]
[1 0]
Fertilizer_Used    int32
Irrigation_Used    int32
dtype: object


,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare,Region_North,Region_South,Region_West,Soil_Type_Clay,...,Soil_Type_Peaty,Soil_Type_Sandy,Soil_Type_Silt,Crop_Cotton,Crop_Maize,Crop_Rice,Crop_Soybean,Crop_Wheat,Weather_Condition_Rainy,Weather_Condition_Sunny
0,897.077239,27.676966,0,1,122,6.555816,0,0,1,0,...,0,1,0,1,0,0,0,0,0,0
1,992.673282,18.026142,1,1,140,8.527341,0,1,0,1,...,0,0,0,0,0,1,0,0,1,0
2,147.998025,29.794042,0,0,106,1.127443,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,986.866331,16.644190,0,1,146,6.517573,1,0,0,0,...,0,1,0,0,0,0,1,0,1,0
4,730.379174,31.620687,1,1,110,7.248251,0,1,0,0,...,0,0,1,0,0,0,0,1,0,0


In [6]:
x = df.drop('Yield_tons_per_hectare', axis=1)
y = df['Yield_tons_per_hectare']

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=42)
x_train.head()

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest
566853,North,Sandy,Soybean,887.383906,33.050569,True,True,Cloudy,79
382311,South,Clay,Soybean,179.188444,24.699056,False,False,Rainy,88
241519,South,Silt,Wheat,649.310754,30.057577,False,False,Cloudy,120
719220,North,Loam,Cotton,221.489100,19.508065,False,True,Cloudy,101
905718,East,Peaty,Rice,737.016449,18.233438,True,False,Rainy,88


In [75]:
model = RandomForestRegressor()
model.fit(x_train, y_train)

RandomForestRegressor()

In [76]:
model.score(x_test, y_test)

0.9073975994684819

In [79]:
model.predict(x_test)[0:5]

array([4.07722426, 4.70845273, 7.06430923, 2.68872388, 4.01823223])

In [80]:
y_test.head()

987231    3.840988
79954     5.138173
567130    6.401523
500891    2.658805
55399     2.797703
Name: Yield_tons_per_hectare, dtype: float64

In [ ]:


y_pred = model.predict(x_test)

print("R² Score :", round(r2_score(y_test, y_pred), 4))   # how well variance is explained
print("MAE      :", round(mean_absolute_error(y_test, y_pred), 4))  # avg error in original units (tons/hectare)
print("RMSE     :", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))  # penalises large errors more

R² Score : 0.9074
MAE      : 0.4119
RMSE     : 0.5167


In [84]:
import joblib

# Save
joblib.dump(model, 'crop_yield_model.pkl')
print("Model saved!")

# Load back
model = joblib.load('crop_yield_model.pkl')
print("Model loaded!")

Model saved!
Model loaded!


In [85]:
model.score(x_test, y_test)

0.9073975994684819

In [86]:
print("Number of trees    :", model.n_estimators)
print("Total nodes        :", sum(t.tree_.node_count for t in model.estimators_))
print("Avg depth per tree :", sum(t.get_depth() for t in model.estimators_) / model.n_estimators)

Number of trees    : 100
Total nodes        : 94818224
Avg depth per tree : 51.63


In [88]:
model = RandomForestRegressor(
    n_estimators     = 100,
    max_depth        = 20,        # hard cap — down from 51
    min_samples_leaf = 5,         # no leaf smaller than 5 samples
    min_samples_split= 10,        # no split unless 10+ samples present
    max_features     = 'sqrt',
    random_state     = 42,
    n_jobs           = -1
)
model.fit(x_train, y_train)

# Check nodes and depth again
print("Total nodes :", sum(t.tree_.node_count for t in model.estimators_))
print("Avg depth   :", sum(t.get_depth() for t in model.estimators_) / model.n_estimators)
print("R²          :", round(model.score(x_test, y_test), 4))

Total nodes : 9869008
Avg depth   : 20.0
R²          : 0.9083


In [89]:

import os



joblib.dump(model, 'crop_yield_final.pkl', compress=3)

size = os.path.getsize('crop_yield_final.pkl') / (1024*1024)
print(f"Final model size: {size:.2f} MB")

Final model size: 228.10 MB


In [90]:
m = joblib.load('crop_yield_final.pkl')

In [91]:
m.score(x_test, y_test)

0.9082699589090735

In [94]:
len(x_test)
len(y_test)

len(x_train)

750000

In [95]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/101.7 MB 2.6 MB/s eta 0:00:39
    --------------------------------------- 1.8/101.7 MB 3.4 MB/s eta 0:00:30
   - -------------------------------------- 3.1/101.7 MB 4.0 MB/s eta 0:00:25
   - -------------------------------------- 4.7/101.7 MB 4.8 MB/s eta 0:00:21
   -- ------------------------------------- 6.3/101.7 MB 5.5 MB/s eta 0:00:18
   --- ------------------------------------ 8.4/101.7 MB 6.1 MB/s eta 0:00:16
   ---- ----------------------------------- 10.5/101.7 MB 6.7 MB/s eta 0:00:14
   ---- ----------------------------------- 12.6/101.7 MB 7.1 MB/s eta 0:00:13
   ----- ---------------------------------- 12.8/101.7 MB 7.1 MB/s eta 0:00:13
   ----- ---------------------------------- 14.2/101.7 MB 6.4 MB/s eta 0:00:14
   ------- -------------------------------- 17.8/101.7 MB 7.4 MB/s eta 0:0

In [ ]:


model_xgb = XGBRegressor(
    n_estimators  = 300,
    max_depth     = 6,       # default is 6 — already well constrained
    learning_rate = 0.05,
    subsample     = 0.8,     # uses 80% of data per tree — reduces overfitting
    colsample_bytree = 0.8,  # uses 80% of features per tree
    random_state  = 42,
    n_jobs        = -1
)

model_xgb.fit(
    x_train, y_train,
    eval_set         = [(x_test, y_test)],
    verbose          = 50    # prints progress every 50 trees
)

[0]	validation_0-rmse:1.63170
[50]	validation_0-rmse:0.53844
[100]	validation_0-rmse:0.50216
[150]	validation_0-rmse:0.50138
[200]	validation_0-rmse:0.50140
[250]	validation_0-rmse:0.50144
[299]	validation_0-rmse:0.50148


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [97]:
y_pred = model_xgb.predict(x_test)

print("R² Score :", round(r2_score(y_test, y_pred), 4))
print("MAE      :", round(mean_absolute_error(y_test, y_pred), 4))
print("RMSE     :", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))

# Check overfitting
print("\nTrain R²:", round(model_xgb.score(x_train, y_train), 4))
print("Test  R²:", round(model_xgb.score(x_test,  y_test),  4))

R² Score : 0.9128
MAE      : 0.4
RMSE     : 0.5015

Train R²: 0.9138
Test  R²: 0.9128


In [99]:
# # XGBoost has its own save method — much more efficient than joblib
# model_xgb.save_model('crop_yield_xgb.json')

# Load back
model_xgb1 = XGBRegressor()
model_xgb1.load_model('crop_yield_xgb.json')

# size = os.path.getsize('crop_yield_xgb.json') / (1024*1024)
# print(f"Model size: {size:.2f} MB")

In [100]:
y_pred1 = model_xgb1.predict(x_test)

print("R² Score :", round(r2_score(y_test, y_pred1), 4))
print("MAE      :", round(mean_absolute_error(y_test, y_pred1), 4))
print("RMSE     :", round(np.sqrt(mean_squared_error(y_test, y_pred1)), 4))

# Check overfitting
print("\nTrain R²:", round(model_xgb1.score(x_train, y_train), 4))
print("Test  R²:", round(model_xgb1.score(x_test,  y_test),  4))

R² Score : 0.9128
MAE      : 0.4
RMSE     : 0.5015

Train R²: 0.9138
Test  R²: 0.9128


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ('cat',  OneHotEncoder(
                handle_unknown='ignore',  # avoids errors on unseen categories in new data
                sparse_output=False       # returns a regular array instead of sparse matrix
             ), cat_cols),
    ('num',  StandardScaler(), num_cols),
    ('bool', 'passthrough',    bool_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators     = 1000,
        max_depth        = 6,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        random_state     = 42,
        n_jobs           = -1
    ))
])



In [10]:
pipeline.fit(x_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
y_pred = pipeline.predict(x_test)

print("R² Score :", round(r2_score(y_test, y_pred), 4))
print("MAE      :", round(mean_absolute_error(y_test, y_pred), 4))
print("RMSE     :", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))

R² Score : 0.9126
MAE      : 0.4004
RMSE     : 0.5018


In [12]:
import joblib

# Save everything in one file
joblib.dump(pipeline, 'please_fkingwork.pkl', compress=3)
print("Pipeline saved!")



Pipeline saved!


In [10]:
import pickle

# Save model
with open("mode_final_ithink.pkl", "wb") as file:
    pickle.dump(pipeline, file)

In [1]:
pip install scikit-learn==1.7.2


   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.7 MB 4.2 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.7 MB 4.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.7 MB 4.9 MB/s eta 0:00:02
   ------------------- -------------------- 4.2/8.7 MB 5.5 MB/s eta 0:00:01
   ------------------------- -------------- 5.5/8.7 MB 5.7 MB/s eta 0:00:01
   --------------------------------- ------ 7.3/8.7 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 6.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sklearn
print(sklearn.__version__)

1.7.2
